# Jani — entrenamiento del clasificador de hojas de café

**Cómo usarlo (no hay que editar nada):**
1. En Colab: `Archivo → Subir notebook` y elige este archivo.
2. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar`.
3. `Entorno de ejecución → Ejecutar todo`. Tarda unos **80–100 minutos** (entrena dos modelos y elige el mejor).
4. Al final el navegador descarga **`jani_model.zip`**. Descomprímelo **dentro de** `app/public/models/arabica-v1/`
   del repositorio, reemplazando lo que haya, y en `app/` ejecuta `npm run check:assets` y `npm run build`.

**Qué hace:** descarga los datos, limpia duplicados, entrena `efficientnet_lite0` y `mobilenetv3_small_100` con JMuBEN
(Kenia) + BRACOL (Brasil) + una parte de Uganda (fotos de campo, repartidas por hoja), calibra la confianza y el umbral
de "no estoy segura" con fotos de campo, prueba en hojas que nunca vio (Uganda) y en un país que nunca vio (**Perú**),
exporta a ONNX, cuantiza a int8 y se queda con el mejor modelo estable que pese menos de 10 MB.

**Qué trae el zip:** un solo archivo `.onnx` (el recomendado), `model_card.json` (clases, normalización, temperatura,
umbrales, métricas y fuentes) y `confusion_*.png` para el video.

Si una descarga de Mendeley falla, el mensaje dice qué página abrir y con qué nombre subir el `.zip` a Colab (ícono de
carpeta, a la izquierda); luego `Ejecutar todo` otra vez. Si Uganda/Perú no se pueden bajar, el modelo igual se entrena,
pero sin el número honesto de precisión fuera de distribución.

In [ ]:
!pip -q install timm datasets onnx onnxruntime onnxscript scikit-learn
!apt-get -qq install -y unrar libarchive-tools > /dev/null 2>&1 || true

## Configuración
No hace falta cambiar nada. Los valores están pensados para una GPU T4 de Colab gratis.

In [ ]:
# Modelos que se entrenan en esta ejecución. Al final se elige el mejor automáticamente.
MODELS = ["efficientnet_lite0", "mobilenetv3_small_100"]

IMG_SIZE = 224
EPOCHS = 8
BATCH = 64
LR = 1e-3
# JMuBEN está desbalanceado (Sana 18.983, Minador 16.978 ... Phoma 6.571). Con este tope se usan TODAS las fotos de
# la clase más chica y la misma cantidad de las demás (balanceado). None = todas las fotos (desbalanceado, más lento).
MAX_PER_CLASS = 6500
USE_BRACOL = True        # suma hojas completas de Brasil (BRACOL)
SEED = 42

# Orden canónico de clases. La app depende de este orden.
CLASSES = ["sana", "roya", "minador", "phoma", "cercospora"]

# Regla de "no estoy segura". El umbral de confianza se elige con la calibración de campo (sección 5):
# el más bajo que logre TARGET_PRECISION respondiendo al menos MIN_COVERAGE de las veces.
TARGET_PRECISION = 0.90
MIN_COVERAGE = 0.30
DEFAULT_CONF = 0.70      # se usa si no hay datos externos
MARGIN_THRESHOLD = 0.15

# Fotos de campo. Uganda se reparte POR HOJA (las copias de una misma hoja quedan juntas). Perú nunca se entrena.
UGANDA_SPLIT = {"train": 0.4, "calib": 0.2, "hold": 0.4}   # entrenamiento / calibración / prueba
PERU_SPLIT = {"calib": 0.5, "hold": 0.5}                   # calibración / prueba
FIELD_SHARE = 0.25       # en cada época, 1 de cada 4 fotos es de campo (Uganda + BRACOL)
DUP_THRESHOLD = 0.80     # parecido de textura desde el que dos fotos son copias de la misma hoja (copias >= 0,94; distintas <= 0,49)

MAX_MODEL_MB = 10        # criterio del reto
INT8_MAX_DIFF = 0.02     # int8 aceptable: su precisión media difiere <= 2 puntos de la del fp32...
INT8_MAX_DISAGREE = 0.05 # ...y da una respuesta distinta en <= 5 % de las fotos

OUT_DIR = "jani_model"
SMOKE_TEST = False       # True solo para probar el notebook sin GPU con imágenes sintéticas

In [ ]:
import os, re, io, csv, json, glob, math, random, zipfile, shutil, subprocess, urllib.request, time
from collections import Counter
import numpy as np, torch, torch.nn as nn, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")   # aviso inofensivo con precisión mixta
warnings.filterwarnings("ignore", category=DeprecationWarning)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if SMOKE_TEST:
    EPOCHS, BATCH, MAX_PER_CLASS = 1, 8, None
    matplotlib.use("Agg"); plt.show = lambda *a, **k: None
else:
    assert device == "cuda", "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"
NUM_WORKERS = 0 if SMOKE_TEST else 2

## 1. Datos de entrenamiento: JMuBEN (Kenia)

Espejo en Hugging Face (`Project-AgML/arabica_coffee_leaf_disease_classification`, CC BY 4.0, 58.549 imágenes, ~1,8 GB).

In [ ]:
def to_canonical(name):
    """Nombre de carpeta o etiqueta -> clase canónica, o None si no es una de las 5."""
    n = re.sub(r"[^a-z]+", " ", str(name).lower()).strip()
    w = set(n.split())
    if w & {"unhealthy", "diseased", "enferma", "enfermas"}: return None
    if re.search(r"cer?s?cospora|cescospora", n) or ("brown" in w and "eye" in w): return "cercospora"
    if "phoma" in w or ("brown" in w and ("leaf" in w or "spot" in w)): return "phoma"   # BRACOL: "brown leaf spot"
    if w & {"rust", "roya", "clr"}: return "roya"
    if w & {"miner", "minador"}: return "minador"
    if w & {"healthy", "health", "sana", "sano", "sanas", "saludable"}: return "sana"
    return None

if SMOKE_TEST:
    # Imágenes sintéticas: cada clase con un color dominante distinto (solo para probar el código).
    hf_names = ["Cerscospora", "Healthy", "Leaf_rust", "Miner", "Phoma"]
    _cols = [(150, 90, 40), (40, 160, 60), (220, 140, 30), (200, 200, 170), (90, 60, 40)]
    _rng = np.random.default_rng(0)
    def _fake(c):
        a = np.clip(np.array(_cols[c]) + _rng.normal(0, 25, (96, 96, 3)), 0, 255).astype("uint8")
        return Image.fromarray(a)
    _fake_rows = [{"image": _fake(c), "label": c} for c in range(5) for _ in range(16)]
    labels_all = np.array([r["label"] for r in _fake_rows])
    def hf_row(i): return _fake_rows[i]
else:
    from datasets import load_dataset
    ds = load_dataset("Project-AgML/arabica_coffee_leaf_disease_classification", split="train")
    hf_names = ds.features["label"].names
    labels_all = np.array(ds["label"])
    def hf_row(i): return ds[int(i)]

print("Clases en el dataset:", hf_names, "| total:", len(labels_all))
unmapped = [n for n in hf_names if to_canonical(n) is None]
assert not unmapped, f"Clases sin mapear: {unmapped}. Ajusta to_canonical()."
hf_to_idx = {i: CLASSES.index(to_canonical(n)) for i, n in enumerate(hf_names)}
print("Mapeo:", {hf_names[i]: CLASSES[j] for i, j in hf_to_idx.items()})

train_idx, val_idx = [], []
rng = np.random.default_rng(SEED)
for c in range(len(hf_names)):
    idx = rng.permutation(np.where(labels_all == c)[0])
    if MAX_PER_CLASS: idx = idx[:MAX_PER_CLASS]
    n_val = max(1, int(0.15 * len(idx)))
    val_idx += idx[:n_val].tolist(); train_idx += idx[n_val:].tolist()
print("Entrenamiento:", len(train_idx), "| Validación:", len(val_idx))

## 2. Datos externos (Mendeley Data)

- **BRACOL** (Brasil, CC BY 4.0): se suma al entrenamiento. Hojas completas con etiquetas en `leaf/dataset.csv`
  (columna `predominant_stress`: 0 sana, 1 minador, 2 roya, 3 phoma, 4 cercospora). El zip publicado en Mendeley está
  incompleto (le falta el final): se recupera lo que trae con `bsdtar` (~1.400 de 1.747 hojas).
- **Uganda** (CC BY 4.0) y **Perú** (CC BY 4.0): solo para la **prueba fuera de distribución**. El modelo nunca los ve al entrenar.

Se descargan con la API pública de Mendeley (archivo por archivo). Si algo falla, el mensaje dice qué página abrir;
pulsa *Download All* y sube el `.zip` a Colab (ícono de carpeta) con el nombre que indica; luego `Ejecutar todo`.

In [ ]:
MENDELEY = {   # id, versión, página
    "bracol": ("yy2k5y8mxg", 1, "https://data.mendeley.com/datasets/yy2k5y8mxg/1"),
    "uganda": ("k36wnd6knb", 1, "https://data.mendeley.com/datasets/k36wnd6knb/1"),
    "peru":   ("mfpxg4y65r", 1, "https://data.mendeley.com/datasets/mfpxg4y65r/1"),
}
# Perú son fotos de ~2 MB (3,5 GB en total): para la prueba basta una muestra por clase. None = todas.
# (La API de Mendeley lista como máximo 1.000 archivos por carpeta: Uganda queda en ~3.000 de 3.312.)
EXTERNAL_MAX_PER_FOLDER = {"bracol": None, "uganda": None, "peru": 200}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
STRESS_CODES = {0: "sana", 1: "minador", 2: "roya", 3: "phoma", 4: "cercospora"}   # BRACOL predominant_stress
API = "https://data.mendeley.com/public-api/datasets"
# Mendeley rechaza (403) el identificador por defecto de Python: se usa el de un navegador.
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def http_json(url, retries=6):
    # Mendeley a veces responde 503/429 por un momento: se reintenta con espera creciente.
    for k in range(retries):
        try:
            req = urllib.request.Request(url, headers={**UA, "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=60) as r:
                return json.load(r)
        except Exception:
            if k == retries - 1: raise
            time.sleep(min(60, 3 * 2 ** k))

def http_download(url, path, retries=5):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    last = None
    for k in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(path + ".part", "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            os.replace(path + ".part", path); return True
        except Exception as e:
            last = e; time.sleep(3 * 2 ** k)
    print("  No se pudo bajar", os.path.basename(path), "->", last); return False

def unpack(archive, dest):
    # Descomprime zip/rar. Si el zip está incompleto (BRACOL), bsdtar recupera lo que trae.
    if archive.lower().endswith(".rar"):
        os.makedirs(dest, exist_ok=True)
        subprocess.run(["unrar", "x", "-o+", "-inul", archive, dest + "/"])
        return
    try:
        with zipfile.ZipFile(archive) as z: z.extractall(dest)
    except zipfile.BadZipFile:
        print(f"  {os.path.basename(archive)} está incompleto: se recupera lo posible con bsdtar...")
        os.makedirs(dest, exist_ok=True)
        subprocess.run([BSDTAR, "-xf", archive, "-C", dest], stderr=subprocess.DEVNULL)

BSDTAR = "bsdtar"

def extract_nested(folder):
    for _ in range(3):
        found = False
        for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
            if p.lower().endswith((".zip", ".rar")):
                try:
                    unpack(p, os.path.splitext(p)[0]); os.remove(p); found = True
                except Exception as e:
                    print("  No se pudo extraer", p, e)
        if not found: break

def shrink_images(folder, max_side=1024, size=768):
    # Las fotos grandes se reducen (la app igual las lleva a 224x224): baja el tiempo de evaluación.
    from concurrent.futures import ThreadPoolExecutor
    def one(p):
        try:
            with Image.open(p) as im:
                if max(im.size) <= max_side: return
                im = im.convert("RGB"); im.thumbnail((size, size))
            im.save(p, quality=92)
        except Exception:
            pass
    paths = [os.path.join(dp, f) for dp, _, fs in os.walk(folder) for f in fs if f.lower().endswith(IMG_EXT)]
    with ThreadPoolExecutor(8) as ex: list(ex.map(one, paths))

def mendeley_listing(did, ver):
    # [(carpeta relativa, nombre, url, bytes)] de todo el dataset, vía la API pública.
    folders = http_json(f"{API}/{did}/folders/{ver}")
    by_id = {f["id"]: f for f in folders}
    def rel(fid):
        parts = []
        while fid in by_id:
            parts.append(by_id[fid]["name"]); fid = by_id[fid].get("parent_id")
        return os.path.join(*reversed(parts)) if parts else ""
    out = []
    for fid in ["root"] + list(by_id):
        for f in http_json(f"{API}/{did}/files?folder_id={fid}&version={ver}"):
            cd = f["content_details"]
            out.append(("" if fid == "root" else rel(fid), f["filename"], cd["download_url"], cd.get("size") or 0))
    return out

def fetch_mendeley(key):
    did, ver, page = MENDELEY[key]
    dest = f"data/{key}"
    if os.path.isdir(dest) and any(f.lower().endswith(IMG_EXT) for _, _, fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    manual = f"{key}.zip"
    if os.path.exists(manual):                       # subido a mano a Colab
        print(f"Usando {manual} subido a mano"); unpack(manual, dest); extract_nested(dest); shrink_images(dest); return dest
    try:
        files = mendeley_listing(did, ver)
    except Exception as e:
        print(f"  {key}: la API de Mendeley no respondió ({e}).")
        print(f"  Abre {page} -> Download All -> sube el archivo a Colab como '{manual}' y vuelve a ejecutar.")
        return None
    cap = EXTERNAL_MAX_PER_FOLDER.get(key)
    if cap:
        groups = {}
        for f in files: groups.setdefault(f[0], []).append(f)
        files = [f for g in groups.values() for f in random.Random(SEED).sample(g, min(cap, len(g)))]
    archives = [f for f in files if f[1].lower().endswith((".zip", ".rar"))]
    images = [f for f in files if f[1].lower().endswith(IMG_EXT)]
    print(f"Descargando {key}: {len(archives)} archivo(s) comprimido(s) y {len(images)} imágenes, "
          f"{sum(f[3] for f in archives + images) / 1e6:.0f} MB...")
    for rel_dir, name, url, _ in archives:
        path = os.path.join(dest, rel_dir, name)
        if http_download(url, path):
            unpack(path, os.path.join(dest, rel_dir)); os.remove(path)
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(16) as ex:
        ok = sum(ex.map(lambda f: http_download(f[2], os.path.join(dest, f[0], f[1])), images))
    if images: print(f"  {ok}/{len(images)} imágenes descargadas")
    extract_nested(dest)
    shrink_images(dest)
    return dest

def scan_folder(root):
    """(ruta, clase canónica o 'desconocida') usando los nombres de las carpetas."""
    items = []
    for dp, _, fs in os.walk(root):
        parts = os.path.relpath(dp, root).split(os.sep)
        label = next((l for l in (to_canonical(p) for p in reversed(parts)) if l), None)
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                items.append((os.path.join(dp, f), label or "desconocida"))
    return items

def csv_labels(root):
    """Etiquetas de BRACOL (hojas completas): dataset*.csv con columnas id y predominant_stress."""
    by_path = {}
    for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            rows = list(csv.DictReader(open(path, encoding="utf-8-sig")))
        except Exception:
            continue
        if not rows or not {"id", "predominant_stress"} <= set(rows[0].keys()): continue
        folder = os.path.dirname(path)
        img_dirs = [d for d in glob.glob(os.path.join(folder, "**"), recursive=True) if os.path.isdir(d)]
        index = {}
        for d in img_dirs:
            for f in os.listdir(d):
                stem, ext = os.path.splitext(f)
                if ext.lower() in IMG_EXT and os.path.basename(d).lower() in ("leaf", "leaves", "images", "leaf_images"):
                    index[stem] = os.path.join(d, f)
        for r in rows:
            p = index.get(str(r["id"]).strip())
            code = str(r["predominant_stress"]).strip()
            if p and code.isdigit() and int(code) in STRESS_CODES:
                by_path[p] = STRESS_CODES[int(code)]
    return by_path

def summarize(name, items):
    print(f"{name}: {len(items)} imágenes →", dict(Counter(l for _, l in items)))

def make_smoke_external():
    """Copias sintéticas con la misma estructura que los datasets reales (solo SMOKE_TEST)."""
    out = {}
    def save(path, c):
        os.makedirs(os.path.dirname(path), exist_ok=True); _fake(c).save(path)
    names = ["Cerscospora", "Healthy", "Leaf_rust", "Miner", "Phoma"]   # índice = color de _fake
    # BRACOL: hojas completas con CSV + recortes en carpetas 1_health…5_cercospora
    root = "data/smoke_bracol"; shutil.rmtree(root, ignore_errors=True)
    code_to_fake = {0: 1, 1: 3, 2: 2, 3: 4, 4: 0}
    os.makedirs(f"{root}/dataset/leaf", exist_ok=True)
    with open(f"{root}/dataset/dataset.csv", "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["id", "predominant_stress", "miner", "rust", "phoma", "cercospora", "severity"])
        for i in range(10):
            code = i % 5; save(f"{root}/dataset/leaf/{i+1}.jpg", code_to_fake[code]); w.writerow([i + 1, code, 0, 0, 0, 0, 0])
    for k, folder in enumerate(["1_health", "2_miner", "3_rust", "4_phoma", "5_cercospora"]):
        for i in range(3): save(f"{root}/dataset/symptom/train/{folder}/{i}.jpg", code_to_fake[k])
    out["bracol"] = root
    for key, folders in [("uganda", ["healthy", "rust", "phoma"]), ("peru", ["sana", "roya", "ojo de gallo"])]:
        root = f"data/smoke_{key}"; shutil.rmtree(root, ignore_errors=True)
        for folder in folders:
            lab = to_canonical(folder)
            c = names.index({"sana": "Healthy", "roya": "Leaf_rust", "phoma": "Phoma"}[lab]) if lab else 3
            for i in range(12): save(f"{root}/{folder}/{i}.jpg", c)
            shutil.copy(f"{root}/{folder}/0.jpg", f"{root}/{folder}/dup_exacto.jpg")
            with Image.open(f"{root}/{folder}/1.jpg") as im: im.rotate(90).save(f"{root}/{folder}/dup_girada.jpg")
            open(f"{root}/{folder}/vacia.jpg", "wb").close()
        out[key] = root
    return out

smoke_dirs = make_smoke_external() if SMOKE_TEST else {}
ext = {}
for key in MENDELEY:
    if key == "bracol" and not USE_BRACOL: continue
    d = smoke_dirs.get(key) if SMOKE_TEST else fetch_mendeley(key)
    items = scan_folder(d) if d else []
    if d and key == "bracol":
        labels = csv_labels(d)
        items = [(p, labels.get(p, l)) for p, l in items]
        print(f"  BRACOL: {len(labels)} hojas completas etiquetadas desde el CSV")
    ext[key] = items
    summarize(key, items)
print("\nRevisa los conteos. Si una clase conocida aparece como 'desconocida', ajusta to_canonical().")

## 3. Limpieza y reparto por hoja

Uganda trae fotos vacías, duplicados exactos y **copias de la misma hoja** (giradas o con otro tono). Si una copia queda
en entrenamiento y otra en prueba, el resultado se infla. Por eso:

1. Se descartan archivos vacíos o ilegibles y los duplicados exactos (si un duplicado tiene dos etiquetas distintas, se
   descarta).
2. Se agrupan las copias de una misma foto: parecido de textura ≥ `DUP_THRESHOLD` en cualquiera de los 8 giros/espejos
   (un cambio de tono, brillo o compresión no las separa). Medido con los datos reales: copias ≥ 0,94, hojas distintas ≤ 0,49.
3. Se reparte **por hoja y por clase**: Uganda en entrenamiento / calibración / prueba; Perú en calibración / prueba
   (**Perú nunca se usa para entrenar**: es la prueba honesta en un país que el modelo no vio).

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor

def readable(p):
    # Decodifica la imagen completa: verify() deja pasar archivos truncados que luego fallan al entrenar.
    try:
        if os.path.getsize(p) == 0: return False
        with Image.open(p) as im: im.convert("RGB").load()
        return True
    except Exception:
        return False

def file_md5(p):
    with open(p, "rb") as f: return hashlib.md5(f.read()).hexdigest()

def texture_desc(p, size=64):
    """Textura fina de la foto (escala de grises menos su versión desenfocada), sobre el cuadrado central.
    Medido con datos reales: copias de una misma foto (giradas, espejadas, con otro tono o recomprimidas) dan
    parecido >= 0,94; hojas distintas, <= 0,49 (Uganda, Perú y BRACOL)."""
    from PIL import ImageFilter
    with Image.open(p) as im:
        g = im.convert("L")
        w, h = g.size; s = min(w, h); l, t = (w - s) // 2, (h - s) // 2
        g = g.crop((l, t, l + s, t + s)).resize((size, size))
        a = np.asarray(g, dtype=np.float32) - np.asarray(g.filter(ImageFilter.GaussianBlur(3)), dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

def near_duplicate_groups(paths, thr=None, block=512):
    """Índice de grupo por foto: misma hoja si el parecido es >= thr en algún giro o espejo."""
    thr = DUP_THRESHOLD if thr is None else thr
    with ThreadPoolExecutor(8) as ex: descs = list(ex.map(texture_desc, paths))
    norm = float(np.sqrt(descs[0].size))
    D = np.stack([d.flatten() for d in descs]) / norm                     # vectores de norma 1
    parent = list(range(len(paths)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for k in range(4):
        for flip in (False, True):
            V = np.stack([(np.fliplr(np.rot90(d, k)) if flip else np.rot90(d, k)).flatten() for d in descs]) / norm
            for s in range(0, len(paths), block):
                ii, jj = np.where(V[s:s + block] @ D.T >= thr)
                for i, j in zip(ii + s, jj):
                    if i != j:
                        a, b = find(i), find(j)
                        if a != b: parent[a] = b
    return [find(i) for i in range(len(paths))]

def clean_and_group(name, items):
    """[(ruta, etiqueta)] -> [(ruta, etiqueta, grupo)] sin ilegibles, sin duplicados y con la hoja como grupo."""
    with ThreadPoolExecutor(16) as ex: ok = list(ex.map(readable, [p for p, _ in items]))
    bad = ok.count(False)
    items = [it for it, k in zip(items, ok) if k]
    with ThreadPoolExecutor(16) as ex: hashes = list(ex.map(file_md5, [p for p, _ in items]))
    by_hash = {}
    for it, h in zip(items, hashes): by_hash.setdefault(h, []).append(it)
    kept, exact, contradictory = [], 0, 0
    for group in by_hash.values():
        if len({l for _, l in group}) > 1: contradictory += len(group); continue
        kept.append(group[0]); exact += len(group) - 1
    groups = near_duplicate_groups([p for p, _ in kept]) if kept else []
    labels_of = {}
    for (_, l), g in zip(kept, groups): labels_of.setdefault(g, set()).add(l)
    out = [(p, l, f"{name}:{g}") for (p, l), g in zip(kept, groups) if len(labels_of[g]) == 1]
    contradictory += len(kept) - len(out)
    n_leaves = len({g for _, _, g in out})
    print(f"{name}: {bad} vacías/ilegibles, {exact} duplicados exactos, {contradictory} con etiqueta contradictoria "
          f"-> {len(out)} fotos de {n_leaves} hojas distintas")
    return out

def group_split(items, fractions, seed=SEED):
    """Reparte [(ruta, etiqueta, grupo)] por hoja y por clase según las proporciones (en cantidad de fotos)."""
    r = random.Random(seed)
    names = list(fractions)
    cum = np.cumsum([fractions[k] for k in names]) / sum(fractions.values())
    out = {k: [] for k in names}
    by_label = {}
    for p, l, g in items: by_label.setdefault(l, {}).setdefault(g, []).append((p, l))
    for l in sorted(by_label):
        leaves = [by_label[l][g] for g in sorted(by_label[l])]; r.shuffle(leaves)
        total, done = sum(len(x) for x in leaves), 0
        for leaf in leaves:
            pos = (done + len(leaf) / 2) / total
            out[names[min(int(np.searchsorted(cum, pos, side="right")), len(names) - 1)]] += leaf
            done += len(leaf)
    return out

bracol_known = []
if ext.get("bracol"):
    bracol_known = [(p, l) for p, l, _ in clean_and_group("bracol", [(p, l) for p, l in ext["bracol"] if l in CLASSES])]

field_train = []          # fotos de campo que sí entran al entrenamiento (parte de Uganda)
ood = {}                  # por país: calib / hold (clases conocidas) y unknown_calib / unknown_hold (clases que no conoce)
for key, split in [("uganda", UGANDA_SPLIT), ("peru", PERU_SPLIT)]:
    if not ext.get(key): continue
    cleaned = clean_and_group(key, ext[key])
    known = [it for it in cleaned if it[1] in CLASSES]
    unknown = [(p, "desconocida", g) for p, l, g in cleaned if l not in CLASSES]
    parts = group_split(known, split)
    uparts = group_split(unknown, {"calib": 0.5, "hold": 0.5}) if unknown else {"calib": [], "hold": []}
    field_train += parts.get("train", [])
    # Las clases desconocidas llevan una etiqueta ficticia: solo se mide si la app dice "no estoy segura".
    ood[key] = {"calib": parts.get("calib", []), "hold": parts.get("hold", []),
                "unknown_calib": [(p, "sana") for p, _ in uparts["calib"]],
                "unknown_hold": [(p, "sana") for p, _ in uparts["hold"]]}
    print(f"  {key}: entrenamiento {len(parts.get('train', []))} | calibración {len(ood[key]['calib'])} | "
          f"prueba {len(ood[key]['hold'])} | desconocidas {len(uparts['calib'])}+{len(uparts['hold'])}")

print("\nBRACOL para entrenar:", len(bracol_known), dict(Counter(l for _, l in bracol_known)))
print("Fotos de campo de Uganda para entrenar:", len(field_train), dict(Counter(l for _, l in field_train)))
if not ood:
    print("SIN datos externos: la temperatura queda en 1, el umbral en DEFAULT_CONF y no habrá número honesto (sección 2).")

## 4. Datasets y aumentación

La app estira la foto completa a 224×224 sin recortar y normaliza con la media y desviación del modelo. La evaluación hace
exactamente lo mismo; el entrenamiento agrega recortes, giros, cambios de color, perspectiva y desenfoque. En cada época,
`FIELD_SHARE` de las fotos son de campo (Uganda + BRACOL), para que el modelo no aprenda solo el estilo de JMuBEN.

In [ ]:
def make_transforms(mean, std):
    train_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.6, 1.67)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
        T.RandomPerspective(0.2, p=0.3),
        T.ColorJitter(0.35, 0.35, 0.3, 0.05),
        T.RandomApply([T.GaussianBlur(5)], p=0.2),
        T.ToTensor(), T.Normalize(mean, std),
    ])
    eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf

class HFSet(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        r = hf_row(self.idx[i])
        return self.tf(r["image"].convert("RGB")), hf_to_idx[r["label"]]

class FileSet(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, l = self.items[i]
        with Image.open(p) as im:          # ya se verificó que todas se leen (sección 3)
            return self.tf(im.convert("RGB")), CLASSES.index(l)

def make_train_loader(train_tf):
    jm = HFSet(train_idx, train_tf)
    field = bracol_known + field_train
    if not field:
        return DataLoader(jm, BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    data = torch.utils.data.ConcatDataset([jm, FileSet(field, train_tf)])
    w = [(1 - FIELD_SHARE) / len(jm)] * len(jm) + [FIELD_SHARE / len(field)] * len(field)
    sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(data),
                                                     replacement=True, generator=torch.Generator().manual_seed(SEED))
    return DataLoader(data, BATCH, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True)

## 5. Entrenar, calibrar, evaluar y exportar cada modelo

Para cada modelo: entrenamiento → **temperatura calibrada con fotos de campo** (calibración de Uganda + Perú, cada país
pesa igual) → umbral de duda elegido con esas mismas fotos → métricas en la **prueba** (fotos que no se usaron ni para
entrenar ni para calibrar) → ONNX fp32 → int8 → elección del archivo.

In [ ]:
CONF_GRID = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

def loader(items, tf): return DataLoader(FileSet(items, tf), BATCH, num_workers=NUM_WORKERS)

def get_logits(m, dl):
    m.eval(); L, Y = [], []
    with torch.no_grad():
        for x, y in dl:
            L.append(m(x.to(device)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def rule(logits, T, conf, margin=MARGIN_THRESHOLD):
    p = torch.softmax(logits / T, 1)
    top2 = p.topk(2, 1).values
    sure = (top2[:, 0] >= conf) & ((top2[:, 0] - top2[:, 1]) >= margin)
    return p.argmax(1), sure

def stats(logits, y, T, conf):
    pred, sure = rule(logits, T, conf)
    cov = sure.float().mean().item()
    prec = (pred[sure] == y[sure]).float().mean().item() if sure.any() else None
    return {"n": int(len(y)), "accuracy_all": (pred == y).float().mean().item(), "coverage": cov,
            "accuracy_when_answering": prec, "escalation_rate": 1 - cov}

def show(name, s):
    prec = f"{s['accuracy_when_answering']:.3f}" if s["accuracy_when_answering"] is not None else "-"
    print(f"  {name}: precisión total {s['accuracy_all']:.3f} | responde en {s['coverage']:.1%} | "
          f"precisión cuando responde {prec} | n={s['n']}")

def abstention(logits, T, conf):
    return 1 - rule(logits, T, conf)[1].float().mean().item()

def fit_temperature(groups):
    """groups: [(logits, y)] por país. Minimiza la entropía cruzada dando el mismo peso a cada país."""
    L = torch.cat([g[0] for g in groups]); Y = torch.cat([g[1] for g in groups])
    W = torch.cat([torch.full((len(g[1]),), 1.0 / len(g[1])) for g in groups]) / len(groups)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = (nn.functional.cross_entropy(L / logT.exp(), Y, reduction="none") * W).sum()
        loss.backward(); return loss
    opt.step(closure)
    T = float(logT.detach().exp())
    return min(max(T, 0.05), 20.0) if math.isfinite(T) else 1.0

def choose_conf(calib, unknown, T):
    """Umbral más bajo con precisión media >= TARGET_PRECISION y cobertura media >= MIN_COVERAGE (cada país pesa igual)."""
    rows = []
    for c in CONF_GRID:
        per = [stats(lg, yy, T, c) for lg, yy in calib.values()]
        cov = float(np.mean([s["coverage"] for s in per]))
        prec = float(np.mean([s["accuracy_when_answering"] or 0.0 for s in per]))
        abst = float(np.mean([abstention(lg, T, c) for lg in unknown])) if unknown else None
        rows.append((c, cov, prec, abst))
    print("  umbral | responde | precisión cuando responde | dice 'no estoy segura' ante clases desconocidas")
    for c, cov, prec, abst in rows:
        print(f"   {c:.2f}  |  {cov:6.1%}  |  {prec:.3f}  |  {'-' if abst is None else f'{abst:.1%}'}")
    ok = [r for r in rows if r[1] >= MIN_COVERAGE and r[2] >= TARGET_PRECISION]
    if ok: return ok[0][0]
    covered = [r for r in rows if r[1] >= MIN_COVERAGE]
    print(f"  Ningún umbral logra precisión {TARGET_PRECISION} con cobertura {MIN_COVERAGE}: se usa el de mayor precisión.")
    return max(covered, key=lambda r: r[2])[0] if covered else DEFAULT_CONF

def plot_cm(out, name, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(len(CLASSES))))
    fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm, cmap="Greens")
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)): ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title(name)
    plt.tight_layout(); plt.savefig(f"{out}/confusion_{name}.png", dpi=120); plt.show(); plt.close(fig)

def ort_session(path):
    # Se evalúa SIN optimizaciones de grafo: así da los mismos números que onnxruntime-web (WASM) en el teléfono.
    # Con optimizaciones, el onnxruntime de PC fusiona el int8 en kernels propios y puede dar otros resultados
    # (medido: hasta 5 logits de diferencia con int8 por canal; sin optimizar coincide con la app en < 0,5).
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def onnx_logits(path, dataset, n=None):
    sess = ort_session(path)
    idx = list(range(len(dataset))) if not n or n >= len(dataset) else random.Random(SEED).sample(range(len(dataset)), n)
    L, Y, t = [], [], 0.0
    for i in idx:
        x, y = dataset[i]; t0 = time.time()
        L.append(sess.run(None, {"input": x.unsqueeze(0).numpy()})[0][0]); t += time.time() - t0; Y.append(y)
    return torch.tensor(np.array(L)), torch.tensor(Y), 1000 * t / max(1, len(idx))

def run_model(name):
    print(f"\n==================== {name} ====================")
    out = f"{OUT_DIR}/{name}"; os.makedirs(out, exist_ok=True)
    model = timm.create_model(name, pretrained=True, num_classes=len(CLASSES)).to(device)
    cfg = timm.data.resolve_data_config({}, model=model)
    mean, std = [float(v) for v in cfg["mean"]], [float(v) for v in cfg["std"]]
    print("Normalización:", mean, std)
    train_tf, eval_tf = make_transforms(mean, std)
    train_dl = make_train_loader(train_tf)
    val_ds = HFSet(val_idx, eval_tf)
    val_dl = DataLoader(val_ds, BATCH, num_workers=NUM_WORKERS)
    calib_items = {k: v["calib"] for k, v in ood.items() if v["calib"]}

    # --- Entrenamiento: se guarda la época con mejor precisión en la calibración de campo (si existe) ---
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best = -1.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        lg, yy = get_logits(model, val_dl)
        val_acc = (lg.argmax(1) == yy).float().mean().item()
        field = [(lambda r: (r[0].argmax(1) == r[1]).float().mean().item())(get_logits(model, loader(v, eval_tf)))
                 for v in calib_items.values()]
        sel = float(np.mean(field)) if field else val_acc
        print(f"Época {ep+1}/{EPOCHS} | pérdida {tot/len(train_dl):.3f} | validación JMuBEN {val_acc:.4f} | "
              f"calibración de campo {'-' if not field else f'{sel:.4f}'} | {time.time()-t0:.0f}s")
        if sel >= best:
            best = sel; torch.save(model.state_dict(), f"{out}/best.pt")
    model.load_state_dict(torch.load(f"{out}/best.pt", map_location=device)); model.eval()

    # --- Calibración con fotos de campo ---
    val_logits, val_y = get_logits(model, val_dl)
    calib = {k: get_logits(model, loader(v, eval_tf)) for k, v in calib_items.items()}
    unknown_calib = [get_logits(model, loader(v["unknown_calib"], eval_tf))[0] for v in ood.values() if v["unknown_calib"]]
    if calib:
        temperature = fit_temperature(list(calib.values()))
        print("Temperatura (calibrada con fotos de campo):", round(temperature, 3))
        print("Elección del umbral (calibración de campo, cada país pesa igual):")
        conf = choose_conf(calib, unknown_calib, temperature)
    else:
        temperature, conf = 1.0, DEFAULT_CONF
        print("Sin fotos de campo: temperatura 1 y umbral por defecto (la validación de JMuBEN está inflada y no sirve para calibrar).")
    print("Umbral de confianza elegido:", conf)

    # --- Métricas en la PRUEBA (fotos no usadas para entrenar ni calibrar) ---
    metrics = {"in_distribution_val": stats(val_logits, val_y, temperature, conf)}
    show("Validación JMuBEN (inflada, no representativa)", metrics["in_distribution_val"])
    plot_cm(out, "validacion", val_y.numpy(), val_logits.argmax(1).numpy())
    for key, parts in ood.items():
        if parts["hold"]:
            lg, yy = get_logits(model, loader(parts["hold"], eval_tf))
            metrics[f"ood_{key}_holdout"] = stats(lg, yy, temperature, conf)
            note = "parte de Uganda se usó para entrenar; estas hojas no" if key == "uganda" and field_train else "nunca vista"
            show(f"{key} - prueba ({note})", metrics[f"ood_{key}_holdout"])
            plot_cm(out, key, yy.numpy(), lg.argmax(1).numpy())
            present = sorted(set(yy.tolist()))
            print(classification_report(yy, lg.argmax(1), labels=present, target_names=[CLASSES[i] for i in present], zero_division=0))
        if parts["unknown_hold"]:
            lg, _ = get_logits(model, loader(parts["unknown_hold"], eval_tf))
            rate = abstention(lg, temperature, conf)
            metrics[f"ood_{key}_unknown_class_abstention"] = {"n": len(parts["unknown_hold"]), "abstention_rate": rate}
            print(f"  {key}: ante {len(parts['unknown_hold'])} fotos de clases que no conoce, dice 'no estoy segura' en {rate:.1%} (ideal ~100%)")

    # --- ONNX fp32 + int8 ---
    import onnx
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
    model_cpu = model.to("cpu").eval()
    fp32, int8 = f"{out}/model_fp32.onnx", f"{out}/model_int8.onnx"
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
    try:
        torch.onnx.export(model_cpu, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model_cpu, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, **kw)
    onnx.checker.check_model(onnx.load(fp32))

    class Calib(CalibrationDataReader):
        # Calibración de la cuantización con fotos de JMuBEN y de campo (las que el modelo ve en entrenamiento).
        def __init__(self, n=300):
            rr = random.Random(SEED)
            xs = [HFSet([i], eval_tf)[0][0] for i in rr.sample(train_idx, min(n * 2 // 3, len(train_idx)))]
            field = bracol_known + field_train
            xs += [FileSet([f], eval_tf)[0][0] for f in rr.sample(field, min(n // 3, len(field)))] if field else []
            self.it = iter([{"input": x.unsqueeze(0).numpy()} for x in xs])
        def get_next(self): return next(self.it, None)

    int8_ok = True
    try:
        pre = f"{out}/_pre.onnx"
        try:
            from onnxruntime.quantization.shape_inference import quant_pre_process
            quant_pre_process(fp32, pre)
        except Exception:
            shutil.copy(fp32, pre)
        quantize_static(pre, int8, Calib(), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
        os.remove(pre)
    except Exception as e:
        int8_ok = False; print("La cuantización falló:", e)

    # Evaluación de los ONNX en la prueba de cada país (o en validación si no hay datos de campo).
    eval_sets = {k: FileSet(v["hold"], eval_tf) for k, v in ood.items() if v["hold"]} or {"val": val_ds}
    sizes, per, preds = {}, {}, {}
    for tag, path in [("fp32", fp32), ("int8", int8)]:
        if tag == "int8" and not int8_ok: continue
        sizes[tag] = round(os.path.getsize(path) / 1e6, 2)
        per[tag], preds[tag] = {}, {}
        for key, dset in eval_sets.items():
            lg, yy, ms = onnx_logits(path, dset, n=None if key != "val" else 1500)
            s = stats(lg, yy, temperature, conf); s["ms_per_image_cpu_unoptimized"] = round(ms, 1)
            per[tag][key] = s; preds[tag][key] = lg.argmax(1); metrics[f"onnx_{tag}_{key}"] = s
            show(f"ONNX {tag} {key} ({sizes[tag]} MB, {ms:.0f} ms/foto en CPU sin optimizar)", s)
    mean_acc = {tag: float(np.mean([s["accuracy_all"] for s in d.values()])) for tag, d in per.items()}
    stable = False
    if int8_ok:
        # Estable = misma precisión media (±INT8_MAX_DIFF) Y casi siempre la misma respuesta que el fp32.
        diff = abs(mean_acc["fp32"] - mean_acc["int8"])
        disagree = float(np.mean([(preds["fp32"][k] != preds["int8"][k]).float().mean().item() for k in preds["fp32"]]))
        stable = diff <= INT8_MAX_DIFF and disagree <= INT8_MAX_DISAGREE
        metrics["int8_vs_fp32"] = {"accuracy_diff": diff, "prediction_disagreement": disagree, "stable": stable}
        print(f"  int8 vs fp32: diferencia de precisión media {diff:.3f} (máx. {INT8_MAX_DIFF}), "
              f"respuestas distintas en {disagree:.1%} de las fotos (máx. {INT8_MAX_DISAGREE:.0%})")
    if stable and sizes["int8"] <= MAX_MODEL_MB: rec = "int8"
    elif sizes["fp32"] <= MAX_MODEL_MB: rec = "fp32"
    else: rec = None
    if int8_ok and not stable:
        print("  La cuantización int8 cambia demasiado las respuestas: no se usa.")
    if rec is None:
        print(f"  {name} queda descartado: el fp32 pesa {sizes['fp32']} MB (> {MAX_MODEL_MB}) y el int8 no es confiable.")
    print(f"Precisión media por país: fp32 {mean_acc['fp32']:.3f}" + (f", int8 {mean_acc['int8']:.3f}" if int8_ok else "")
          + f" -> recomendado: {rec}")
    os.remove(f"{out}/best.pt")
    return {"name": name, "out": out, "mean": mean, "std": std, "temperature": temperature, "conf": conf,
            "metrics": metrics, "sizes": sizes, "recommended": rec, "per_country": per.get(rec, {}) if rec else {},
            "score": mean_acc[rec] if rec else -1.0, "score_source": "promedio de países" if ood else "validación"}

In [ ]:
results = []
for name in MODELS:
    try:
        results.append(run_model(name))
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"\n*** {name} falló: {e}. Se sigue con el siguiente modelo. ***")
    if device == "cuda": torch.cuda.empty_cache()

keys = list(ood) or ["val"]
print("\nRESUMEN (prueba, aritmética del teléfono)")
print("modelo | archivo | MB | " + " | ".join(keys) + " | promedio | 'no estoy segura' ante desconocidas | umbral | temperatura")
for r in results:
    pc = r["per_country"]
    cols = " | ".join(f"{pc[k]['accuracy_all']:.3f}" if k in pc else "-" for k in keys)
    ab = [v["abstention_rate"] for k, v in r["metrics"].items() if k.endswith("unknown_class_abstention")]
    mb = r["sizes"].get(r["recommended"], "-") if r["recommended"] else "-"
    print(f"{r['name']} | {r['recommended']} | {mb} | {cols} | {r['score']:.3f} | "
          f"{'-' if not ab else f'{np.mean(ab):.1%}'} | {r['conf']} | {r['temperature']:.3f}")

## 6. Elegir el mejor modelo, escribir la ficha y descargar

Gana el de **mayor precisión media por país** (cada país pesa igual) entre los que pesan ≤ 10 MB y cuyo archivo es estable;
si empatan (±0,5 puntos), el más liviano.

In [ ]:
def clean(x):
    """NaN/Infinity -> None para que model_card.json sea JSON válido."""
    if isinstance(x, float) and not math.isfinite(x): return None
    if isinstance(x, dict): return {k: clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [clean(v) for v in x]
    return x

valid = [r for r in results if r["recommended"]]
assert valid, "Ningún modelo quedó por debajo de 10 MB con un archivo estable, o todos fallaron. Revisa los mensajes de arriba."
top = max(r["score"] for r in valid)
best = min([r for r in valid if r["score"] >= top - 0.005], key=lambda r: r["sizes"][r["recommended"]])
print("Elegido:", best["name"], best["recommended"], f"(precisión media por país {best['score']:.3f})")

final = f"{OUT_DIR}/final"; shutil.rmtree(final, ignore_errors=True); os.makedirs(final)
model_file = f"model_{best['recommended']}.onnx"
shutil.copy(f"{best['out']}/{model_file}", f"{final}/{model_file}")
for png in glob.glob(f"{best['out']}/confusion_*.png"): shutil.copy(png, final)

split_info = {k: {"calib": len(v["calib"]), "hold": len(v["hold"]),
                  "unknown_calib": len(v["unknown_calib"]), "unknown_hold": len(v["unknown_hold"])} for k, v in ood.items()}
card = {
    "id": f"arabica-v1-{best['name']}",
    "architecture": best["name"],
    "classes": CLASSES,
    "input": {"name": "input", "shape": [1, 3, IMG_SIZE, IMG_SIZE], "layout": "NCHW", "color": "RGB",
              "resize": "estirar a 224x224 (sin recorte)", "scale": "dividir entre 255", "mean": best["mean"], "std": best["std"]},
    "output": {"name": "logits", "postprocess": "softmax(logits / temperature)"},
    "temperature": best["temperature"],
    "unsure_rule": {"min_confidence": best["conf"], "min_margin": MARGIN_THRESHOLD},
    "files": {best["recommended"]: model_file},
    "recommended_file": model_file,
    "size_mb": {best["recommended"]: best["sizes"][best["recommended"]]},
    "metrics": best["metrics"],
    "candidates": [{"model": r["name"], "file": r["recommended"], "size_mb": r["sizes"], "score_mean_by_country": r["score"],
                    "per_country_accuracy": {k: v["accuracy_all"] for k, v in r["per_country"].items()},
                    "min_confidence": r["conf"], "temperature": r["temperature"]} for r in results],
    "training": {"epochs": EPOCHS, "img_size": IMG_SIZE, "max_per_class_jmuben": MAX_PER_CLASS, "seed": SEED,
                 "field_share_per_epoch": FIELD_SHARE if (bracol_known or field_train) else 0,
                 "uganda_split": UGANDA_SPLIT, "peru_split": PERU_SPLIT, "duplicate_threshold": DUP_THRESHOLD,
                 "external_split_sizes": split_info,
                 "calibration": "temperatura y umbral con la calibración de campo (Uganda + Perú, cada país pesa igual)",
                 "threshold_rule": f"umbral más bajo con precisión media ≥ {TARGET_PRECISION} y cobertura media ≥ {MIN_COVERAGE}",
                 "model_selection": f"mayor precisión media por país en la prueba; int8 solo si su precisión media difiere ≤ {INT8_MAX_DIFF} del fp32 y responde distinto en ≤ {INT8_MAX_DISAGREE:.0%} de las fotos"},
    "training_data": [
        {"name": "JMuBEN / JMuBEN2", "country": "Kenia", "license": "CC BY 4.0",
         "source": "https://huggingface.co/datasets/Project-AgML/arabica_coffee_leaf_disease_classification",
         "doi": ["10.17632/t2r6rszp5c.1", "10.17632/tgv3zb82nd.1"], "images_used": len(train_idx)},
        {"name": "BRACOL", "country": "Brasil", "license": "CC BY 4.0",
         "source": "https://data.mendeley.com/datasets/yy2k5y8mxg/1", "images_used": len(bracol_known)},
        {"name": "Uganda (Soroti University), parte de entrenamiento", "country": "Uganda", "license": "CC BY 4.0",
         "source": "https://data.mendeley.com/datasets/k36wnd6knb/1", "images_used": len(field_train)},
    ],
    "evaluation_data": [
        {"name": "Uganda (Soroti University)", "source": "https://data.mendeley.com/datasets/k36wnd6knb/1",
         "note": "calibración y prueba con hojas distintas a las de entrenamiento (reparto por hoja)"},
        {"name": "Perú (Saposoa)", "source": "https://data.mendeley.com/datasets/mfpxg4y65r/1",
         "note": "nunca se usa para entrenar: es la prueba en un país no visto"},
    ],
    "known_limits": [
        "Solo hojas de arábica; no cubre frutos ni robusta.",
        "JMuBEN son hojas recortadas y con aumentación: la validación interna está inflada.",
        "El número honesto es 'ood_peru_holdout' (país nunca visto); 'ood_uganda_holdout' son hojas no vistas de un país que sí se usó en parte para entrenar.",
        "No reconoce deficiencias nutricionales ni enfermedades fuera de las 5 clases; ante ellas puede no abstenerse.",
    ],
}
if SMOKE_TEST: card["smoke_test"] = True
json.dump(clean(card), open(f"{final}/model_card.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, allow_nan=False)

zip_path = shutil.make_archive("jani_model", "zip", final)
print(json.dumps(clean({"metrics": card["metrics"], "candidates": card["candidates"]}), indent=2, ensure_ascii=False))
print("\nListo:", zip_path, "-> descomprimir dentro de app/public/models/arabica-v1/")
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Descarga manual desde el panel de archivos:", zip_path)